In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline

In [ ]:
wine = pd.read_csv('assets/wines.csv')

In [ ]:
wine.head()

In [ ]:
wine.set_index('date', inplace=True)
wine.index = pd.to_datetime(wine.index)

In [ ]:
wine.plot()

In [ ]:
from src.tde import UnivariateTDE

In [ ]:
# desta vez fazemos a divisão no tempo
train_end = pd.Timestamp('1993-12-01')
test_start = pd.Timestamp('1994-01-01')

In [ ]:
training, testing, mean_value = {}, {}, {}
for col in wine.columns:
    # últimos 6 meses para validação
    train = wine.loc[:train_end, col]
    test = wine.loc[test_start:, col]

    # normalização com dados de TREINO!
    mean_value[col] = train.mean()
    train /= mean_value[col]
    test /= mean_value[col]

    # serie temporal como matriz
    training[col] = UnivariateTDE(train, horizon=1, k=4)
    testing[col] = UnivariateTDE(test, horizon=1, k=4)

In [ ]:
training['Sparkling'].head()

In [ ]:
# concatenando as matrizes
train_df = pd.concat(training, axis=0)
train_df.head()

In [ ]:
train_df.tail()

In [ ]:
testing['Sparkling'].head()

In [ ]:
X_train = train_df.drop('t+1', axis=1)
y_train = train_df['t+1']

In [ ]:
from sklearn.linear_model import Ridge
# treinando o modelo global
global_model = Ridge()
global_model.fit(X_train, y_train)

In [ ]:
# previsão
X_test = testing['Sparkling'].drop('t+1', axis=1)
global_forecasts = global_model.predict(X_test)
# revertendo a normalização
global_forecasts *= mean_value['Sparkling']

In [ ]:
X_train_local = training['Sparkling'].drop('t+1', axis=1)
y_train_local = training['Sparkling']['t+1']

# treinando o modelo local
local_model = Ridge()
local_model.fit(X_train_local, y_train_local)

In [ ]:
# previsão
X_test = testing['Sparkling'].drop('t+1', axis=1)
local_forecasts = local_model.predict(X_test)

local_forecasts *= mean_value['Sparkling']

In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
mean_absolute_error(testing['Sparkling']['t+1'], global_forecasts)

In [ ]:
mean_absolute_error(testing['Sparkling']['t+1'], local_forecasts)